# Flood AutoML — Deteksi Anomali Cuaca untuk Estimasi Probabilitas Banjir
**Balikpapan & Samarinda** · Random Forest · XGBoost · CatBoost · Isolation Forest

Mengikuti alur pada jurnal, saya menggabungkan catatan kejadian banjir BNPB dengan data cuaca harian
Open-Meteo, mengubahnya menjadi fitur *time-series* (lag t-1..t-7 dan akumulasi 3/7/14 hari),
menambahkan skor anomali dari **Isolation Forest**, menyeimbangkan kelas dengan **SMOTE**, lalu
melatih tiga algoritma *tree-ensemble* di dalam kerangka **AutoML** untuk memperkirakan probabilitas
banjir dalam **1-3 hari ke depan** (`Flood_next_3d`). Kota di luar Kaltim dipakai sepenuhnya sebagai
data latih agar model mengenali kondisi banjir dan non-banjir.

## 0. Clone Repository (Google Colab)

Saat berjalan di Colab, saya meng-clone repository agar dataset dan skrip pendukung ikut tersedia,
lalu berpindah ke direktori repo sebagai direktori kerja.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/Noelsip/flood-bpn-smd.git"

def in_colab():
    return "google.colab" in sys.modules

if in_colab():
    repo_dir = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(repo_dir)
    print("Repo di-clone. Direktori kerja:", os.getcwd())
else:
    print("Bukan Colab: memakai file repo lokal.")

## 1. Instalasi Dependensi

Saya memasang paket yang dibutuhkan secara otomatis bila belum tersedia.

In [ ]:
import importlib

REQUIRED = ["numpy", "pandas", "scikit-learn", "imbalanced-learn",
            "xgboost", "catboost", "flaml[automl]", "matplotlib"]

def ensure(module, pip_name=None):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or module], check=True)

if in_colab():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *REQUIRED], check=True)
    print("Dependensi terpasang.")
else:
    for module, pip_name in [("flaml", "flaml[automl]"), ("xgboost", "xgboost"),
                             ("catboost", "catboost"), ("imblearn", "imbalanced-learn")]:
        ensure(module, pip_name)
    print("Dependensi lokal siap.")

## 2. Import Library

Saya mengumpulkan seluruh library di satu tempat agar mudah ditelusuri.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix, classification_report)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from flaml import AutoML

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

## 3. Konfigurasi & Daftar Fitur

Saya menetapkan parameter global dan menyusun daftar fitur. `HORIZON = 3` berarti target memuat
kejadian banjir kapan saja dalam **1-3 hari ke depan**, `TRAIN_RATIO = 0.80` membagi data Kaltim
80/20, dan `MIN_TEST_SIZE = 200` menjaga data uji tetap memadai.

In [ ]:
SEED = 42
HORIZON = 3
TARGET = "Flood_next_3d"
ANOMALY_CONTAMINATION = "auto"
TIME_BUDGET = 120
THRESHOLD_BETA = 2.0
TRAIN_RATIO = 0.80
MIN_TEST_SIZE = 200
KALTIM_CITIES = ["Kota Balikpapan", "Kota Samarinda"]

ROOT = Path.cwd()
DATA_PATH = ROOT / "processed" / "dataset_timeseries.csv"
OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(exist_ok=True)

WEATHER_COLS = ["precip", "tmax", "tmin", "wind", "rain", "soil"]
LAG_COLS = [f"{c}_lag{k}" for c in WEATHER_COLS for k in range(1, 8)]
ROLL_COLS = ["rain_roll3_sum", "rain_roll7_sum", "rain_roll14_sum",
             "precip_roll3_sum", "precip_roll7_sum",
             "soil_roll3_mean", "soil_roll7_mean", "tmax_roll7_mean"]
PCTL_COLS = ["rain7_pctl", "precip7_pctl", "soil7_pctl"]
CALENDAR_COLS = ["doy_sin", "doy_cos", "month_sin", "month_cos"]
BASE_FEATURES = LAG_COLS + ROLL_COLS + PCTL_COLS + CALENDAR_COLS

## 4. Pemuatan Dataset

Saya memuat dataset *time-series* siap model lalu memisahkan kota Kaltim (Balikpapan & Samarinda)
dari kota lain. Kota lain nantinya seluruhnya menjadi data latih, sedangkan Kaltim dibagi untuk
latih dan uji.

In [ ]:
data = pd.read_csv(DATA_PATH, parse_dates=["time"])
kaltim = data[data["city"].isin(KALTIM_CITIES)].sort_values("time").reset_index(drop=True)
external = data[~data["city"].isin(KALTIM_CITIES)].reset_index(drop=True)
print("Seluruh dataset:", data.shape)
print("Kaltim         :", kaltim.shape, "| banjir:", int(kaltim[TARGET].sum()))
print("Luar Kaltim    :", external.shape, "| banjir:", int(external[TARGET].sum()))
print(data.groupby("city")[TARGET].agg(hari="count", banjir="sum"))
data.head()

## 5. Exploratory Data Analysis (EDA)

Saya menelaah seberapa timpang kelasnya dan apakah akumulasi hujan benar-benar membedakan hari
banjir dari hari aman, sebagai pembenaran langkah-langkah berikutnya.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
data[TARGET].value_counts().sort_index().plot.bar(ax=ax[0], color=["#4C9F70", "#D1495B"], rot=0)
ax[0].set_title("Sebaran target (0 = aman, 1 = banjir 1-3 hari)")
ax[0].set_xlabel(TARGET)
for label, grp in data.groupby(TARGET):
    ax[1].hist(grp["rain_roll7_sum"], bins=40, alpha=0.6, density=True, label=f"target={label}")
ax[1].set_title("Akumulasi hujan 7 hari per kelas")
ax[1].set_xlabel("rain_roll7_sum"); ax[1].legend()
plt.tight_layout(); plt.show()

## 6. Pembagian Data (Train/Test Split)

Saya menempatkan seluruh kota luar Kaltim ke data latih agar model belajar pola banjir dan
non-banjir dari banyak wilayah, lalu membagi data Kaltim 80% latih dan 20% uji secara kronologis.
Data uji murni Kaltim terbaru dan dijaga minimal 200 baris.

In [ ]:
cut = int(len(kaltim) * TRAIN_RATIO)
kaltim_train = kaltim.iloc[:cut]
test_df = kaltim.iloc[cut:].reset_index(drop=True)
train_df = pd.concat([external, kaltim_train], ignore_index=True)
y_train = train_df[TARGET].astype(int)
y_test = test_df[TARGET].astype(int)

assert len(test_df) >= MIN_TEST_SIZE, f"Data uji {len(test_df)} < {MIN_TEST_SIZE}"
print(f"Train: {train_df.shape} | banjir: {int(y_train.sum())}"
      f" (luar Kaltim + Kaltim <= {kaltim_train['time'].max().date()})")
print(f"Test : {test_df.shape} | banjir: {int(y_test.sum())}"
      f" | rentang {test_df['time'].min().date()} s/d {test_df['time'].max().date()}")

## 7. Deteksi Anomali — Isolation Forest

Saya melatih Isolation Forest pada cuaca data latih, lalu memberi skor anomali untuk setiap baris.
Skor ini menjadi fitur tambahan bagi AutoML—mengikuti gagasan jurnal bahwa kondisi cuaca yang
menyimpang sering mendahului banjir.

In [ ]:
iso = IsolationForest(n_estimators=400, contamination=ANOMALY_CONTAMINATION, random_state=SEED, n_jobs=-1)
iso.fit(train_df[WEATHER_COLS])

train_df["anomaly_score"] = -iso.score_samples(train_df[WEATHER_COLS])
test_df["anomaly_score"] = -iso.score_samples(test_df[WEATHER_COLS])
train_df["is_anomaly"] = (iso.predict(train_df[WEATHER_COLS]) == -1).astype(int)
test_df["is_anomaly"] = (iso.predict(test_df[WEATHER_COLS]) == -1).astype(int)

FEATURES = BASE_FEATURES + ["anomaly_score"]
print("Jumlah fitur model:", len(FEATURES), "(termasuk anomaly_score)")
print("Anomali pada train:", int(train_df["is_anomaly"].sum()),
      "| di antaranya berlabel banjir:", int(train_df.loc[y_train == 1, "is_anomaly"].sum()))

fig, ax = plt.subplots(figsize=(7, 4))
for label, grp in train_df.groupby(TARGET):
    ax.hist(grp["anomaly_score"], bins=40, alpha=0.6, density=True, label=f"target={label}")
ax.set_title("Skor anomali Isolation Forest per kelas (train)")
ax.set_xlabel("anomaly_score"); ax.legend(); plt.tight_layout(); plt.show()

## 8. Standardisasi Fitur

Saya mengepas `StandardScaler` hanya pada data latih lalu menerapkannya ke data uji, agar setiap
fitur berada pada skala yang sebanding dan statistik penskalaan tidak bocor dari masa depan.

In [ ]:
scaler = StandardScaler().fit(train_df[FEATURES])
X_train = pd.DataFrame(scaler.transform(train_df[FEATURES]), columns=FEATURES)
X_test = pd.DataFrame(scaler.transform(test_df[FEATURES]), columns=FEATURES)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)

## 9. Penanganan Ketidakseimbangan Kelas (SMOTE)

Saya menyeimbangkan data latih dengan SMOTE sehingga jumlah hari banjir dan aman menjadi setara.
SMOTE membuat contoh sintetis kelas minoritas dan hanya diterapkan pada data latih agar tidak ada
kebocoran ke data uji.

In [ ]:
print("Sebelum SMOTE:", dict(pd.Series(y_train).value_counts().sort_index()))
X_res, y_res = SMOTE(random_state=SEED).fit_resample(X_train, y_train)
print("Sesudah SMOTE:", dict(pd.Series(y_res).value_counts().sort_index()))

## 10. Pemodelan AutoML (Random Forest, XGBoost, CatBoost)

Saya melatih Random Forest, XGBoost, dan CatBoost pada data yang sudah seimbang, lalu membiarkan
FLAML mencari kombinasi estimator dan hyperparameter terbaik di antara ketiganya dengan metrik
AUC-PR (`ap`).

In [ ]:
def fit_models(X, y):
    fitted = {}
    fitted["Random Forest"] = RandomForestClassifier(
        n_estimators=400, random_state=SEED, n_jobs=-1).fit(X, y)
    fitted["XGBoost"] = XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.03, subsample=0.85, colsample_bytree=0.85,
        reg_lambda=3.0, min_child_weight=3, eval_metric="logloss",
        random_state=SEED, n_jobs=-1).fit(X, y)
    fitted["CatBoost"] = CatBoostClassifier(
        iterations=400, depth=6, learning_rate=0.05,
        random_seed=SEED, verbose=0, allow_writing_files=False).fit(X, y)

    automl = AutoML()
    automl.fit(X_train=X, y_train=y, task="classification", metric="ap",
               estimator_list=["rf", "xgboost", "catboost"], time_budget=TIME_BUDGET,
               eval_method="cv", n_splits=5, seed=SEED, verbose=1)
    fitted["AutoML (FLAML)"] = automl
    return fitted, automl

models, automl = fit_models(X_res, y_res)
print("Estimator AutoML terbaik:", automl.best_estimator)
print("Konfigurasi terbaik:", automl.best_config)

## 11. Evaluasi Model

Saya membandingkan keluaran tiap model dengan accuracy, precision, recall, F1, AUC-ROC, dan AUC-PR.
Isolation Forest ikut disertakan sebagai *baseline* tak-terawasi: skornya untuk kurva, flag
`is_anomaly` untuk prediksi kelasnya.

In [ ]:
def proba(model, X):
    return np.asarray(model.predict_proba(X))[:, 1]

scores = {name: proba(m, X_test) for name, m in models.items()}
scores["Isolation Forest"] = test_df["anomaly_score"].to_numpy()

rows = []
for name, p in scores.items():
    pred = test_df["is_anomaly"].to_numpy() if name == "Isolation Forest" else (p >= 0.5).astype(int)
    rows.append(dict(Model=name,
                     Accuracy=accuracy_score(y_test, pred),
                     Precision=precision_score(y_test, pred, zero_division=0),
                     Recall=recall_score(y_test, pred, zero_division=0),
                     F1=f1_score(y_test, pred, zero_division=0),
                     AUC_ROC=roc_auc_score(y_test, p),
                     AUC_PR=average_precision_score(y_test, p)))
comparison = pd.DataFrame(rows).set_index("Model").sort_values("AUC_PR", ascending=False)
comparison.to_csv(OUT_DIR / "model_comparison.csv")
comparison

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for name, p in scores.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    ax[0].plot(fpr, tpr, lw=2, label=f"{name} ({roc_auc_score(y_test, p):.3f})")
    pr, rc, _ = precision_recall_curve(y_test, p)
    ax[1].plot(rc, pr, lw=2, label=f"{name} ({average_precision_score(y_test, p):.3f})")
ax[0].plot([0, 1], [0, 1], "k--", lw=1)
ax[0].set_title("ROC"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].legend()
ax[1].set_title("Precision-Recall"); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].legend()
plt.tight_layout(); plt.show()

## 12. Pemilihan Ambang Operasional

Saya memilih ambang dari prediksi model AutoML pada data latih dengan memaksimalkan F-beta (beta=2)
agar lebih sensitif menangkap kejadian banjir, lalu menerapkannya ke data uji.

In [ ]:
best_name = "AutoML (FLAML)"
best_model = models[best_name]
p_train = proba(best_model, X_train)
beta2 = THRESHOLD_BETA ** 2

candidates = np.unique(np.r_[np.linspace(0.05, 0.95, 181), p_train])
grid = []
for thr in candidates:
    pred = (p_train >= thr).astype(int)
    prec = precision_score(y_train, pred, zero_division=0)
    rec = recall_score(y_train, pred, zero_division=0)
    fbeta = (1 + beta2) * prec * rec / (beta2 * prec + rec + 1e-12)
    grid.append((thr, prec, rec, fbeta))
thr_df = pd.DataFrame(grid, columns=["thr", "precision", "recall", "fbeta"])
OPER_THR = float(thr_df.sort_values(["fbeta", "recall"], ascending=False).iloc[0]["thr"])
print(f"Ambang operasional terpilih (F-beta={THRESHOLD_BETA}, utamakan recall): {OPER_THR:.3f}")

## 13. Kinerja Akhir pada Data Uji

Saya mengukur kinerja model operasional pada data uji menggunakan ambang yang terpilih.

In [ ]:
p_test = proba(best_model, X_test)
pred_test = (p_test >= OPER_THR).astype(int)
print(f"Model operasional: {best_name} (AutoML -> {automl.best_estimator})")
print(classification_report(y_test, pred_test, target_names=["Aman", "Banjir"], zero_division=0))
print(f"AUC-ROC = {roc_auc_score(y_test, p_test):.3f} | AUC-PR = {average_precision_score(y_test, p_test):.3f}")

cm = confusion_matrix(y_test, pred_test)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Aman", "Banjir"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Aman", "Banjir"])
ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual")
ax.set_title(f"Confusion Matrix @ {OPER_THR:.3f}")
for (r, c), v in np.ndenumerate(cm):
    ax.text(c, r, str(v), ha="center", va="center", color="white" if v > cm.max() / 2 else "black")
plt.tight_layout(); plt.show()

## 14. Feature Importance & Penyimpanan Hasil

Saya merata-ratakan kepentingan fitur dari ketiga model untuk melihat apakah `anomaly_score` dan
akumulasi hujan benar-benar berkontribusi, lalu menyimpan perbandingan model dan prediksi akhir.

In [ ]:
importances = {
    "Random Forest": pd.Series(models["Random Forest"].feature_importances_, index=FEATURES),
    "XGBoost": pd.Series(models["XGBoost"].feature_importances_, index=FEATURES),
    "CatBoost": pd.Series(models["CatBoost"].feature_importances_, index=FEATURES),
}
imp_df = pd.DataFrame(importances)
imp_df["mean"] = imp_df.mean(axis=1)
imp_df = imp_df.sort_values("mean", ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
imp_df["mean"].head(20).iloc[::-1].plot.barh(ax=ax, color="#3B7DD8")
ax.set_title("20 fitur terpenting (rata-rata 3 model)")
plt.tight_layout(); plt.show()

imp_df.to_csv(OUT_DIR / "feature_importance.csv")

predictions = test_df[["city", "time", TARGET]].copy()
predictions["flood_proba"] = p_test
predictions["anomaly_score"] = test_df["anomaly_score"].to_numpy()
predictions["alert"] = pred_test
predictions.to_csv(OUT_DIR / "flood_predictions.csv", index=False)

for name in ["model_comparison.csv", "feature_importance.csv", "flood_predictions.csv"]:
    print("Disimpan:", OUT_DIR / name)
predictions.head()